In [1]:
!pip install tensorflow keras

  Using cached tensorflow-2.21.0-cp312-cp312-manylinux_2_27_x86_64.whl.metadata (4.4 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-manylinux2010_x86_64.whl.metadata (5.2 kB)
  Using cached h5py-3.14.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (2.7 kB)
  Using cached wheel-0.46.3-py3-none-any.whl.metadata (2.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.6/572.6 MB 904.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 129.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 97.4 MB/s eta 0:00:00
  Attempting uninstall: h5py
    Found existing installation: h5py 3.16.0
    Uninstalling h5py-3.16.0:
      Successfully 

### **1. Setup and Loading the Dataset**

First, we import the necessary libraries and load the CIFAR-10 dataset.

In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models, datasets

# Load the CIFAR-10 dataset
(train_images, train_labels), (test_images, test_labels) = datasets.cifar10.load_data()

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


### **2. Normalizing the Images**

We scale the pixel values to a range of 0 to 1 to help the neural network train faster and more effectively.

In [3]:
# Normalize pixel values to be between 0 and 1
train_images = train_images / 255.0
test_images = test_images / 255.0

### **3. Building the Simple CNN**

This architecture strictly follows your requested structure: exactly 4 Convolutional layers, 1 MaxPooling layer, and 1 Fully Connected (Dense) layer for the output.

In [4]:
model = models.Sequential()

# 4 Convolution Layers
model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)))
model.add(layers.Conv2D(64, (3, 3), activation='relu'))
model.add(layers.Conv2D(64, (3, 3), activation='relu'))
model.add(layers.Conv2D(128, (3, 3), activation='relu'))

# 1 MaxPooling Layer
model.add(layers.MaxPooling2D((2, 2)))

# Flatten the output to feed into the Dense layer
model.add(layers.Flatten())

# 1 Fully Connected (Dense) Layer (Output layer for the 10 classes)
model.add(layers.Dense(10, activation='softmax'))

# View the model architecture
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 30, 30, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 28, 28, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 26, 26, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 24, 24, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 12, 12, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 18432)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │       184,330 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 314,506 (1.20 MB)

 Trainable params: 314,506 (1.20 MB)

 Non-trainable params: 0 (0.00 B)

### **4. Training and Displaying Accuracy**

Finally, we compile the model, train it on the training data, and evaluate its accuracy against the test set.

In [5]:
# Compile the model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Train the model (set to 10 epochs for standard testing)
history = model.fit(
    train_images, train_labels,
    epochs=10,
    validation_data=(test_images, test_labels)
)

# Evaluate the model to display final accuracy
test_loss, test_acc = model.evaluate(test_images, test_labels, verbose=2)
print(f"\\nFinal Test Accuracy: {test_acc:.4f}")

Epoch 1/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 232s 148ms/step - accuracy: 0.5137 - loss: 1.3694 - val_accuracy: 0.6252 - val_loss: 1.0781
Epoch 2/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 230s 147ms/step - accuracy: 0.6643 - loss: 0.9687 - val_accuracy: 0.6837 - val_loss: 0.9252
Epoch 3/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 232s 148ms/step - accuracy: 0.7291 - loss: 0.7885 - val_accuracy: 0.7149 - val_loss: 0.8400
Epoch 4/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 231s 148ms/step - accuracy: 0.7687 - loss: 0.6654 - val_accuracy: 0.7127 - val_loss: 0.8559
Epoch 5/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 232s 148ms/step - accuracy: 0.8080 - loss: 0.5553 - val_accuracy: 0.7103 - val_loss: 0.8835
Epoch 6/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 232s 148ms/step - accuracy: 0.8383 - loss: 0.4610 - val_accuracy: 0.7007 - val_loss: 0.9881
Epoch 7/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 232s 149ms/step - accuracy: 0.8644 - loss: 0.3842 - val_accuracy: 0.7062 - val_loss: 1.0789
Epoch 8/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 232s 148ms/step - ac